# Kiểm toán Dữ liệu & Thống kê Ứng dụng: Kami Ely TikTok Shop (231 Ngày)

**Tác giả:** Ma Văn Hùng · Fresher Data Analyst / Business Analyst  
**Phạm vi dữ liệu:** 10 tệp xuất ước tính bên thứ ba FastMoss (09/03 – 25/10/2026 · 231 ngày · 23.364 dòng thô), đã giả danh hóa định danh creator bằng SHA-256 (`@shop_official`, `@koc_sat01`).  
**Stack:** `pandas`, `numpy`, `scipy.stats`

---
### Nội dung kiểm định trong sổ tay
1. **Nạp & chuẩn hóa dữ liệu 231 ngày:** Bộ parser chuẩn hóa hậu tố tiếng Việt (`k`, `m`, `Tr`, `Tỷ`), khử trùng video theo khóa phức hợp, hợp nhất cờ Ads song ngữ (`YES/có`) và đối soát tổng GMV/sản lượng.
2. **Độ nhạy giao cắt `MA7 < MA28` dưới Winsorization (P99, P95, thay đỉnh 16/06):** Kiểm chứng tín hiệu cảnh báo sớm không phụ thuộc vào một ngày ngoại lai đơn lẻ.
3. **Phân tổ thời lượng phiên Live chính chủ (`n = 600`) & Kiểm định GPM Live vs Video:** So sánh hiệu suất giờ giữa `<2h` và Mega-Live `>10h` kèm cảnh báo nhân quả ngược.
4. **Hồi quy Panel Fixed-Effects trên 10 SKU (`SKU_FE + DOW_FE`):** Ước lượng trên cửa sổ hoạt động sau mở bán (`n = 1.422`, `Within R² = 0,302`) kèm đối chiếu toàn mẫu (`n = 1.760`, `Within R² = 0,490`) với 400 lần cluster bootstrap.
5. **Tập trung mạng lưới KOC (HHI) & Kiểm định Video bật Ads vs Organic (Mann-Whitney U, Fisher Exact):** Tách bạch HHI Live ngoài (`2.152` gồm vệ tinh / `761` độc lập) với danh bạ KOL 90N (`3.842` toàn bảng / `1.471` loại chính chủ còn vệ tinh / `246` thuần ngoài) và kiểm định 4.796 video.

In [1]:
import hashlib
import os
import re
import pandas as pd
import numpy as np
from scipy import stats

DATA_DIR = os.environ.get('KAMI_DATA_DIR', os.path.join(os.getcwd(), 'data', 'kami'))
OFFICIAL_SHA256 = '2884bd95e154a3255b1c0272bcfe9505211a293710db406701023782eb0645ec'
SATELLITE_SHA256 = '6aa6d7c1d040cfe8c061b3e7d7700969504d16b70fe6e65a45d3b573ef71f7bb'

def hash_handle(val) -> str:
    return hashlib.sha256(str(val).strip().lower().encode('utf-8')).hexdigest()

def parse_telemetry_num(val) -> float:
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return 0.0
    if isinstance(val, (int, float, np.integer, np.floating)):
        return float(val)
    s = str(val).replace('₫', '').replace('VNĐ', '').replace('đ', '').replace('%', '').strip()
    if not s or s == '-':
        return 0.0
    low = s.lower()
    mult = 1.0
    for suffix, factor in [('tỷ', 1e9), ('ty', 1e9), ('tr', 1e6), ('m', 1e6), ('k', 1e3)]:
        if low.endswith(suffix):
            mult = factor
            s = s[: -len(suffix)].strip()
            break
    # Handle Vietnamese decimal comma (e.g. '3,37 k') vs thousands separator (e.g. '1,234,567')
    if ',' in s and '.' not in s:
        parts = s.split(',')
        if len(parts) == 2 and len(parts[1]) <= 2:
            s = parts[0] + '.' + parts[1]
        else:
            s = s.replace(',', '')
    elif '.' in s and ',' in s:
        s = s.replace('.', '').replace(',', '.') if s.rfind(',') > s.rfind('.') else s.replace(',', '')
    try:
        return float(s) * mult
    except ValueError:
        return 0.0

def to_num(series: pd.Series) -> pd.Series:
    return series.map(parse_telemetry_num).astype(float)

# 1. Nạp các tệp hợp nhất 231 ngày (09/03 - 25/10/2026)
df_store = pd.read_excel(os.path.join(DATA_DIR, '[Kami Ely]_Store_Daily_Overview_231_Days_Merged.xlsx'))
df_aff_live = pd.read_excel(os.path.join(DATA_DIR, '[Kami Ely]_Affiliate_Livestreams_Merged.xlsx'))
df_cr_live = pd.read_excel(os.path.join(DATA_DIR, '[Kami Ely]_Creator_Livestreams_Merged.xlsx'))
df_aff_vid_raw = pd.read_excel(os.path.join(DATA_DIR, '[Kami Ely]_Affiliate_Videos_Merged.xlsx'))
df_cr_vid = pd.read_excel(os.path.join(DATA_DIR, '[Kami Ely]_Creator_Videos_Merged.xlsx'))
df_sku10 = pd.read_excel(os.path.join(DATA_DIR, '[Kami Ely]_10_SKU_Daily_Timeseries_Merged.xlsx'))
df_prod = pd.read_excel(os.path.join(DATA_DIR, '[Kami Ely]_Shop_Products_Merged.xlsx'))

# Khử trùng lặp khóa phức hợp cho Affiliate_Videos (4.797 -> 4.796 clip)
df_aff_vid = df_aff_vid_raw.drop_duplicates(
    subset=['KOC ID', 'Tiêu Đề Video', 'Thời Gian Xuất Bản']
).copy()

df_store['dt'] = pd.to_datetime(df_store['Ngày Ghi Nhận'])
df_store = df_store.sort_values('dt').reset_index(drop=True)
df_store['gmv'] = to_num(df_store['Doanh Thu Trong Ngày (VNĐ)'])
df_store['units'] = to_num(df_store['Lượng Bán Trong Ngày (Sp)'])

total_gmv = df_store['gmv'].sum()
total_units = int(df_store['units'].sum())
asp = total_gmv / total_units

print(f"Store Daily: {len(df_store)} ngày ({df_store['dt'].min():%d/%m/%Y} - {df_store['dt'].max():%d/%m/%Y})")
print(f"Tổng GMV ước tính: {total_gmv / 1e9:.2f} tỷ VNĐ | Sản lượng: {total_units:,} sp | ASP: {asp:,.0f} đ/sp")
print(f"AffLive: {len(df_aff_live):,} phiên | CrLive: {len(df_cr_live):,} phiên | AffVid (dedup): {len(df_aff_vid):,} / {len(df_aff_vid_raw):,} | ShopProd: {len(df_prod)} SKU")

Store Daily: 231 ngày (09/03/2026 - 25/10/2026)
Tổng GMV ước tính: 15.01 tỷ VNĐ | Sản lượng: 101,896 sp | ASP: 147,298 đ/sp
AffLive: 8,355 phiên | CrLive: 600 phiên | AffVid (dedup): 4,796 / 4,797 | ShopProd: 339 SKU


## 2. Ngoại lai ngày 16/06 & Độ nhạy giao cắt `MA7 < MA28` dưới Winsorization
Kiểm tra Z-score ngày đỉnh (16/06/2026) và đánh giá xem mốc cắt giảm `MA7 < MA28` có bền vững khi cắt ngoại lai ở ngưỡng P99, P95 hoặc thay bằng trung vị tháng 6 hay không.

In [2]:
mu_all = df_store['gmv'].mean()
sd_all = df_store['gmv'].std(ddof=1)
df_store['z_full'] = (df_store['gmv'] - mu_all) / sd_all
peak_row = df_store.loc[df_store['gmv'].idxmax()]
print(f"Ngày đỉnh: {peak_row['dt']:%d/%m/%Y} | GMV = {peak_row['gmv']/1e6:.2f}M VNĐ | Z (full 231d) = {peak_row['z_full']:.2f}")

# Độ nhạy giao cắt MA7 < MA28 giai đoạn hậu đỉnh (>= 10/06)
p99 = df_store['gmv'].quantile(0.99)
p95 = df_store['gmv'].quantile(0.95)
june_med = df_store.loc[df_store['dt'].dt.month == 6, 'gmv'].median()

print(f"Ngưỡng Winsorize: P99 = {p99/1e6:.1f}M | P95 = {p95/1e6:.1f}M | Trung vị T6 = {june_med/1e6:.1f}M")
for label, s in [
    ('Chuỗi gốc (Raw)', df_store['gmv']),
    ('Winsorize P99', df_store['gmv'].clip(upper=p99)),
    ('Winsorize P95', df_store['gmv'].clip(upper=p95)),
    ('Thay 16/06 bằng trung vị T6', df_store['gmv'].where(df_store['dt'] != '2026-06-16', june_med)),
]:
    ma7 = s.rolling(7).mean()
    ma28 = s.rolling(28).mean()
    diff = ma7 - ma28
    cross = df_store[(df_store['dt'] >= '2026-06-10') & (diff < 0) & (diff.shift(1) >= 0)]
    print(f"  {label:<30} -> giao cắt MA7 < MA28: {cross['dt'].dt.strftime('%d/%m/%Y').tolist()}")

Ngày đỉnh: 16/06/2026 | GMV = 315.05M VNĐ | Z (full 231d) = 5.83
Ngưỡng Winsorize: P99 = 176.6M | P95 = 136.8M | Trung vị T6 = 120.5M
  Chuỗi gốc (Raw)                -> giao cắt MA7 < MA28: ['26/06/2026', '22/10/2026']
  Winsorize P99                  -> giao cắt MA7 < MA28: ['26/06/2026', '30/06/2026', '22/10/2026']
  Winsorize P95                  -> giao cắt MA7 < MA28: ['26/06/2026', '01/07/2026', '22/10/2026']
  Thay 16/06 bằng trung vị T6    -> giao cắt MA7 < MA28: ['27/06/2026', '02/07/2026', '22/10/2026']


## 3. Phân tổ thời lượng phiên Live chính chủ (`n = 600`) & Kiểm định GPM Live vs Video
- Phân tích 5 nhóm thời lượng phiên Live chính chủ (`<2h`, `2-4h`, `4-6h`, `6-10h`, `>10h`) — lưu ý hiện tượng nhân quả ngược khi phiên bán tốt thường được kéo dài thời lượng.
- Kiểm định phân phối GPM (nghìn VNĐ / 1.000 lượt xem) giữa Video chính chủ (`n = 276`) và Live chính chủ (`n = 600`) bằng Mann-Whitney U và Welch t-test trên $\log(1 + \text{GPM})$.

In [3]:
df_cr_live['dur_h'] = pd.to_numeric(df_cr_live['Thời Lượng (Giờ)'].astype(str).str.replace(',', '.'), errors='coerce').fillna(0.0)
df_cr_live['gmv'] = to_num(df_cr_live['Doanh Thu Phiên Live (VNĐ)'])
df_cr_live['views'] = to_num(df_cr_live['Tổng Người Xem'])

bins = [-0.01, 2, 4, 6, 10, 999]
labels = ['<2h', '2-4h', '4-6h', '6-10h', '>10h']
df_cr_live['dur_bucket'] = pd.cut(df_cr_live['dur_h'], bins=bins, labels=labels)

summary_dur = df_cr_live.groupby('dur_bucket', observed=False).agg(
    sessions=('gmv', 'size'),
    total_hours=('dur_h', 'sum'),
    total_gmv_b=('gmv', lambda x: x.sum() / 1e9),
).reset_index()
summary_dur['gmv_per_hour_m'] = (summary_dur['total_gmv_b'] * 1000 / summary_dur['total_hours']).round(2)
summary_dur['share_pct'] = (summary_dur['total_gmv_b'] / summary_dur['total_gmv_b'].sum() * 100).round(1)
print("Phân bổ 600 phiên Live chính chủ (@shop_official) theo thời lượng:")
print(summary_dur.to_string(index=False))

# Kiểm định GPM (k VNĐ / 1.000 lượt xem) cấp độ phiên/clip giữa Video & Live chính chủ
df_cr_vid['gmv'] = to_num(df_cr_vid['Doanh Thu Chuyển Đổi (VNĐ)'])
df_cr_vid['views'] = to_num(df_cr_vid['Lượt Xem'])
gpm_vid = df_cr_vid.loc[df_cr_vid['views'] > 0, 'gmv'] / df_cr_vid.loc[df_cr_vid['views'] > 0, 'views']
gpm_live = df_cr_live.loc[df_cr_live['views'] > 0, 'gmv'] / df_cr_live.loc[df_cr_live['views'] > 0, 'views']

t_log, p_log = stats.ttest_ind(np.log1p(gpm_vid), np.log1p(gpm_live), equal_var=False)
u_stat, p_mw = stats.mannwhitneyu(gpm_vid, gpm_live, alternative='two-sided')
print(f"\nGPM Video (n={len(gpm_vid)}): Mean={gpm_vid.mean():.1f}k, Median={gpm_vid.median():.1f}k")
print(f"GPM Live  (n={len(gpm_live)}): Mean={gpm_live.mean():.1f}k, Median={gpm_live.median():.1f}k")
print(f"Welch t (log1p): t = {t_log:.2f}, p = {p_log:.2e} | Mann-Whitney U = {u_stat:,.0f}, p = {p_mw:.2e}")

Phân bổ 600 phiên Live chính chủ (@shop_official) theo thời lượng:
dur_bucket  sessions  total_hours  total_gmv_b  gmv_per_hour_m  share_pct
       <2h       136     211.2358     0.233021            1.10        3.1
      2-4h       232     616.8654     0.940272            1.52       12.4
      4-6h       107     519.2350     1.217567            2.34       16.0
     6-10h        58     462.2862     1.117090            2.42       14.7
      >10h        67     898.6412     4.084845            4.55       53.8

GPM Video (n=276): Mean=90.9k, Median=16.5k
GPM Live  (n=600): Mean=1719.2k, Median=1065.3k
Welch t (log1p): t = -24.39, p = 2.67e-83 | Mann-Whitney U = 12,038, p = 1.44e-92


## 4. Hồi quy Panel Fixed-Effects trên 10 SKU (`SKU_FE + DOW_FE`)
Ước lượng mô hình $\log(1 + \text{units}_{it}) \sim \log(1 + \text{marketing\_lags}_{it}) + \text{SKU\_FE} + \text{DOW\_FE}$ (với biến trễ `lag1_3` là trung bình 3 ngày `t-1..t-3`) cùng 400 lần cluster bootstrap theo cụm SKU. Báo cáo song song **cửa sổ hoạt động sau mở bán** (`n = 1.422`, loại 338 dòng 0đ trước khi SKU lên kệ, `Within R² = 0,302`) và **toàn mẫu gốc** (`n = 1.760`, `Within R² = 0,490`).

In [4]:
df_sku10['dt'] = pd.to_datetime(df_sku10['Ngày Ghi Nhận'])
df_sku10 = df_sku10.sort_values(['Mã SKU', 'dt']).reset_index(drop=True)
df_sku10['units'] = to_num(df_sku10['Lượng Bán Trong Ngày (Sp)'])
df_sku10['live'] = to_num(df_sku10['Livestream Mới Trong Ngày'])
df_sku10['vid'] = to_num(df_sku10['Video Mới Trong Ngày'])
df_sku10['koc'] = to_num(df_sku10['KOC Mới Trong Ngày'])

# Trung bình 3 ngày trễ (t-1..t-3) trong từng cụm SKU, khớp 100% pipeline SSOT
g = df_sku10.groupby('Mã SKU')
df_sku10['live_lag1_3'] = g['live'].transform(lambda s: (s.shift(1) + s.shift(2) + s.shift(3)) / 3.0)
df_sku10['vid_lag1_3'] = g['vid'].transform(lambda s: (s.shift(1) + s.shift(2) + s.shift(3)) / 3.0)
df_sku10['cum_units'] = to_num(df_sku10['Lượng Bán Tích Lũy (Sp)'])
first_active = df_sku10[df_sku10['cum_units'] > 0].groupby('Mã SKU')['dt'].min()
df_sku10['first_active_dt'] = df_sku10['Mã SKU'].map(first_active)
panel_full = df_sku10.dropna(subset=['live_lag1_3', 'vid_lag1_3']).copy()
panel_active = panel_full[panel_full['dt'] >= panel_full['first_active_dt']].copy()

base_cols = ['log_live_t0', 'log_live_lag1_3', 'log_vid_t0', 'log_vid_lag1_3', 'log_koc_t0']

def run_panel_fe(df_in, title):
    p = df_in.copy()
    p['log_units'] = np.log1p(p['units'].clip(lower=0))
    p['log_live_t0'] = np.log1p(p['live'].clip(lower=0))
    p['log_live_lag1_3'] = np.log1p(p['live_lag1_3'].clip(lower=0))
    p['log_vid_t0'] = np.log1p(p['vid'].clip(lower=0))
    p['log_vid_lag1_3'] = np.log1p(p['vid_lag1_3'].clip(lower=0))
    p['log_koc_t0'] = np.log1p(p['koc'].clip(lower=0))
    dow_idx = (p['dt'].dt.dayofweek + 1) % 7
    dow_dummies = pd.DataFrame({f'dow_{d}': (dow_idx == d).astype(float) for d in range(1, 7)})
    work = pd.concat([p[['Mã SKU', 'log_units'] + base_cols].reset_index(drop=True), dow_dummies.reset_index(drop=True)], axis=1)
    X_cols = base_cols + list(dow_dummies.columns)
    dm = work.groupby('Mã SKU')[['log_units'] + X_cols].transform(lambda s: s - s.mean())
    X, y = dm[X_cols].to_numpy(), dm['log_units'].to_numpy()
    beta, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    r2_within = 1.0 - np.sum((y - X @ beta) ** 2) / np.sum((y - y.mean()) ** 2)
    rng = np.random.default_rng(20261101)
    skus = work['Mã SKU'].unique()
    boot = []
    for _ in range(400):
        sampled = rng.choice(skus, size=len(skus), replace=True)
        sub = pd.concat([dm[work['Mã SKU'] == k] for k in sampled])
        b, _, _, _ = np.linalg.lstsq(sub[X_cols].to_numpy(), sub['log_units'].to_numpy(), rcond=None)
        boot.append(b[:len(base_cols)])
    ci_lo, ci_hi = np.quantile(boot, [0.025, 0.975], axis=0)
    print(f"{title} (n = {len(work)}, 10 SKUs, within-SKU R² = {r2_within:.3f}):")
    for col, b, lo, hi in zip(base_cols, beta[:len(base_cols)], ci_lo, ci_hi):
        sig = 'p < 0.05' if (lo > 0 or hi < 0) else 'n.s. (CI chứa 0)'
        print(f"  {col:<18} beta = {b:+.3f} | 95% CI = [{lo:+.3f}, {hi:+.3f}] | {sig}")

run_panel_fe(panel_active, 'Panel FE — Cửa sổ hoạt động sau mở bán')
print()
run_panel_fe(panel_full, 'Đối chiếu toàn mẫu gốc')

Panel FE — Cửa sổ hoạt động sau mở bán (n = 1422, 10 SKUs, within-SKU R² = 0.302):
  log_live_t0        beta = +0.394 | 95% CI = [+0.093, +0.641] | p < 0.05
  log_live_lag1_3    beta = +0.597 | 95% CI = [+0.155, +0.877] | p < 0.05
  log_vid_t0         beta = +0.079 | 95% CI = [-0.410, +0.286] | n.s. (CI chứa 0)
  log_vid_lag1_3     beta = +0.237 | 95% CI = [-0.199, +0.649] | n.s. (CI chứa 0)
  log_koc_t0         beta = +0.165 | 95% CI = [-0.258, +0.755] | n.s. (CI chứa 0)

Đối chiếu toàn mẫu gốc (n = 1760, 10 SKUs, within-SKU R² = 0.490):
  log_live_t0        beta = +0.436 | 95% CI = [+0.057, +0.779] | p < 0.05
  log_live_lag1_3    beta = +0.664 | 95% CI = [+0.299, +0.926] | p < 0.05
  log_vid_t0         beta = +0.213 | 95% CI = [-0.281, +0.513] | n.s. (CI chứa 0)
  log_vid_lag1_3     beta = +0.457 | 95% CI = [-0.062, +0.871] | n.s. (CI chứa 0)
  log_koc_t0         beta = +0.162 | 95% CI = [-0.212, +0.697] | n.s. (CI chứa 0)


## 5. Tập trung Mạng lưới KOC (HHI) & Kiểm định Video Ads vs Organic
Tính chỉ số tập trung `HHI` trên mạng lưới Live ngoài kênh chính chủ (có và không có kênh vệ tinh `@koc_sat01`), đồng thời kiểm định tỷ lệ ra đơn & GPM của Video bật Ads vs Organic trên toàn mẫu (`n = 4.796`). Lưu ý: GPM thấp hơn ở nhóm Ads phản ánh pha loãng tệp hiển thị quảng cáo sang người xem rộng hơn.

In [5]:
# 5A. Chỉ số tập trung HHI trên mạng lưới Live affiliate (231 ngày, PII-safe via SHA-256)
df_aff_live['koc_hash'] = df_aff_live['KOC ID'].map(hash_handle)
df_aff_live['gmv'] = to_num(df_aff_live['Doanh Thu Cho Shop (VNĐ)'])
ext_live = df_aff_live[df_aff_live['koc_hash'] != OFFICIAL_SHA256]

by_koc = ext_live.groupby('koc_hash')['gmv'].sum()
by_koc = by_koc[by_koc > 0]
hhi_ext_all = float(((by_koc / by_koc.sum()) ** 2).sum() * 10_000)

by_indep = by_koc.drop(index=SATELLITE_SHA256, errors='ignore')
hhi_indep = float(((by_indep / by_indep.sum()) ** 2).sum() * 10_000)
print(f"HHI Affiliate Live (gồm @koc_sat01): {hhi_ext_all:,.0f} | Tách @koc_sat01 (độc lập): {hhi_indep:,.0f}")

# 5B. Kiểm định Video bật Ads vs Organic (4.796 clip sau khử trùng)
df_aff_vid['is_ads'] = df_aff_vid['Ủy Quyền Quảng Cáo'].astype(str).str.strip().str.lower().isin(['có', 'yes'])
df_aff_vid['gmv'] = to_num(df_aff_vid['Doanh Thu Đóng Góp (VNĐ)'])
df_aff_vid['views'] = to_num(df_aff_vid['Lượt Xem'])
df_aff_vid['has_order'] = df_aff_vid['gmv'] > 0

ads = df_aff_vid[df_aff_vid['is_ads']]
org = df_aff_vid[~df_aff_vid['is_ads']]
or_val, p_fisher = stats.fisher_exact([
    [int(ads['has_order'].sum()), int((~ads['has_order']).sum())],
    [int(org['has_order'].sum()), int((~org['has_order']).sum())],
])

# Aggregate GPM (k VNĐ / 1.000 lượt xem) và Mann-Whitney U trên nhóm clip phát sinh đơn
agg_gpm_ads = ads['gmv'].sum() / ads['views'].sum()
agg_gpm_org = org['gmv'].sum() / org['views'].sum()

ads_pos = ads[ads['has_order'] & (ads['views'] > 0)]
org_pos = org[org['has_order'] & (org['views'] > 0)]
ads_gpm_pos = ads_pos['gmv'] / ads_pos['views']
org_gpm_pos = org_pos['gmv'] / org_pos['views']
u_pos, p_pos = stats.mannwhitneyu(ads_gpm_pos, org_gpm_pos, alternative='two-sided')

print(f"\nVideo Ads (n={len(ads):,}): Conv = {ads['has_order'].mean()*100:.1f}%, Mean GMV = {ads['gmv'].mean()/1e3:.1f}k đ/clip, Aggregate GPM = {agg_gpm_ads:.1f}k/1k views")
print(f"Video Org (n={len(org):,}): Conv = {org['has_order'].mean()*100:.1f}%, Mean GMV = {org['gmv'].mean()/1e3:.1f}k đ/clip, Aggregate GPM = {agg_gpm_org:.1f}k/1k views (ratio Ads/Org = {agg_gpm_ads/agg_gpm_org:.2f}x)")
print(f"Fisher Exact (tỷ lệ ra đơn): OR = {or_val:.2f}, p = {p_fisher:.2e}")
print(f"Mann-Whitney U trên GPM nhóm clip có đơn: Median Ads = {ads_gpm_pos.median():.1f}k vs Org = {org_gpm_pos.median():.1f}k (p = {p_pos:.2e})")

HHI Affiliate Live (gồm @koc_sat01): 2,152 | Tách @koc_sat01 (độc lập): 761

Video Ads (n=3,775): Conv = 30.3%, Mean GMV = 547.7k đ/clip, Aggregate GPM = 65.7k/1k views
Video Org (n=1,021): Conv = 10.2%, Mean GMV = 125.5k đ/clip, Aggregate GPM = 77.8k/1k views (ratio Ads/Org = 0.84x)
Fisher Exact (tỷ lệ ra đơn): OR = 3.83, p = 4.19e-44
Mann-Whitney U trên GPM nhóm clip có đơn: Median Ads = 116.4k vs Org = 245.3k (p = 3.00e-08)
